# 10. Trajectory analysis of GABAergic interneuron development

This notebook reconstructs developmental trajectories within the GABAergic subset to study how Klf13 knockout affects interneuron maturation and fate decisions. We combine:

1. **Diffusion pseudotime (DPT)** to order cells along a continuous maturation axis.
2. **PAGA (Partition-based Graph Abstraction)** to summarize global connectivity between clusters.
3. **Fate-coloured PAGA** to map PV/SST/Reln/LGE lineage decisions onto the PAGA graph.

All analyses are performed on the GABAergic AnnData object generated in previous notebooks.

## 10.1 Imports, settings, and configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.stats.multitest import multipletests

sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sns.set(style="whitegrid")

# Keys and constants
cluster_key   = "leiden_0.7"
condition_key = "condition"
sample_key    = "batch"
ref_group     = "WT"
test_group    = "KO"

REMOVE_CLUSTER   = "14"
REMOVE_SAMPLE    = "96_6_M_KO"
MIN_CELLS_SAMPLE = 200
ALPHA            = 0.05

# Named clusters and order (from your GABA annotation)
cluster_names = {
    "0":  "mge_prog_prdm16",
    "1":  "cge_prog_nr2f2",
    "2":  "mge_neu_sox6",
    "3":  "lge_neu_ddah1",
    "4":  "mge_neu_st18",
    "5":  "lge_prog_ebf1",
    "6":  "lge_prog_isl1",
    "7":  "mge_prog_lhx6",
    "8":  "mge_neu_erbb4",
    "9":  "mge_prog_sp9",
    "10": "cge_prog_sp8",
    "11": "cge_prog_pax6",
    "12": "mge_prog_nkx21",
    "13": "cge_neu_synpr",
}
cluster_order = [cluster_names[str(i)] for i in range(14) if str(i) in cluster_names]

# Root and terminal clusters for trajectories
ROOT_CLUSTERS    = ["mge_prog_nkx21", "mge_prog_prdm16"]
TERMINAL_CLUSTERS = ["mge_neu_erbb4", "mge_neu_st18",
                     "lge_neu_ddah1", "cge_neu_synpr", "lge_prog_ebf1"]

# WT/KO colors
COND_COLORS = {
    ref_group:  "#378ADD",
    test_group: "#D4537E",
}


def remove_spines(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

## 10.2. Fate map: PV / SST / Reln / LGE groups

At E13.5, Pvalb and Sst are not yet expressed. Fate is inferred from upstream markers:

- **PV fate**: Erbb4, Sox6
- **SST fate**: St18, Prdm16
- **Reln fate**: Nr2f2, Synpr
- **VIP/Reln progenitors**: Sp8, Pax6
- **PV/SST progenitors**: Nkx2-1, Prdm16, Lhx6, Sp9
- **LGE striatal**: Isl1, Ebf1, Ddah1

In [ ]:
# Fate assignment per cluster
FATE_MAP = {
    # MGE — bipotent progenitors (will give PV + SST)
    "mge_prog_nkx21":  "PV/SST progenitor",
    "mge_prog_prdm16": "PV/SST progenitor",
    "mge_prog_lhx6":   "PV/SST progenitor",
    "mge_prog_sp9":    "PV/SST progenitor",
    # MGE — committing to PV (Erbb4+, Sox6+)
    "mge_neu_erbb4":   "PV fate",
    "mge_neu_sox6":    "PV fate",
    # MGE — committing to SST (St18+, Prdm16+)
    "mge_neu_st18":    "SST fate",
    # CGE — committing to Reln (Nr2f2+, Synpr+)
    "cge_prog_nr2f2":  "Reln fate",
    "cge_neu_synpr":   "Reln fate",
    # CGE — VIP/Reln progenitors (Sp8+, Pax6+)
    "cge_prog_sp8":    "VIP/Reln progenitor",
    "cge_prog_pax6":   "VIP/Reln progenitor",
    # LGE — striatal
    "lge_prog_isl1":   "LGE striatal",
    "lge_prog_ebf1":   "LGE striatal",
    "lge_neu_ddah1":   "LGE striatal",
}

FATE_COLORS = {
    "PV fate":             "#E31A1C",   # red
    "PV/SST progenitor":   "#FF7F00",   # orange
    "SST fate":            "#6A3D9A",   # purple
    "Reln fate":           "#1F78B4",   # blue
    "VIP/Reln progenitor": "#33A02C",   # green
    "LGE striatal":        "#B15928",   # brown
}

## 10.3 Load and prepare GABA AnnData

We load the GABA-reclustered object, standardize metadata, remove a small outlier cluster, and exclude samples with too few cells or a problematic batch. Cluster IDs are mapped to descriptive names, and fate groups are added.

In [ ]:
print("Loading data...")
adata = sc.read_h5ad("gaba_reclust_scran0_7.h5ad")

# Clean metadata
adata.obs[cluster_key]   = adata.obs[cluster_key].astype(str)
adata.obs[condition_key] = adata.obs[condition_key].astype(str).str.strip().str.upper()
adata.obs[sample_key]    = adata.obs[sample_key].astype(str)

# Remove unwanted cluster
adata = adata[adata.obs[cluster_key] != REMOVE_CLUSTER].copy()

# Filter samples with enough cells and drop problematic batch
sample_counts = adata.obs[sample_key].value_counts()
keep_samples = [
    s for s in sample_counts[sample_counts >= MIN_CELLS_SAMPLE].index
    if s != REMOVE_SAMPLE
]
adata = adata[adata.obs[sample_key].isin(keep_samples)].copy()

# Map cluster IDs to names and set categorical order
adata.obs["cluster_name"] = (
    adata.obs[cluster_key].map(cluster_names)
    .fillna(adata.obs[cluster_key]).astype(str)
)
adata.obs["cluster_name"] = pd.Categorical(
    adata.obs["cluster_name"],
    categories=cluster_order,
    ordered=True,
)

# Add fate group
adata.obs["fate_group"] = adata.obs["cluster_name"].map(FATE_MAP).fillna("unknown")

print(f"Cells: {adata.n_obs}")
print("Condition counts:")
print(adata.obs[condition_key].value_counts())
print("Fate groups:")
print(adata.obs["fate_group"].value_counts())

## 10.4 Diffusion pseudotime (DPT)

We compute diffusion pseudotime (DPT) to order GABAergic cells along a continuous maturation axis:

1. Compute a neighbor graph and diffusion map.
2. Choose a root cell from Nkx2-1/Prdm16 progenitor clusters.
3. Run `sc.tl.dpt` and visualize pseudotime on UMAP split by condition.

In [ ]:
# Neighbors + diffusion map
if "connectivities" not in adata.obsp:
    print("Computing neighbors...")
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=30, n_pcs=50)
else:
    print("Using existing neighbor graph.")

sc.tl.diffmap(adata, n_comps=15)

# Root cell: in ROOT_CLUSTERS with minimum DC1
root_mask = adata.obs["cluster_name"].isin(ROOT_CLUSTERS)
root_idx = np.where(root_mask)[0]
dc1_vals = adata.obsm["X_diffmap"][root_idx, 1]
root_cell = root_idx[np.argmin(dc1_vals)]

adata.uns["iroot"] = int(root_cell)
print(f"Root cell index: {adata.uns['iroot']}")
print(f"Root cell cluster: {adata.obs['cluster_name'].iloc[root_cell]}")

# DPT
sc.tl.dpt(adata, n_dcs=10)
adata.obs["dpt_pseudotime"] = adata.obs["dpt_pseudotime"]  # rename if needed

print("DPT done.")

In [ ]:
# Preserve original UMAP
umap_orig = np.asarray(adata.obsm["X_umap"].copy())
umap_xy = umap_orig
dpt = adata.obs["dpt_pseudotime"].values

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor="white")

for ax, label, mask in zip(
    axes,
    ["WT", "KO", "All cells"],
    [
        (adata.obs[condition_key] == ref_group),
        (adata.obs[condition_key] == test_group),
        pd.Series(True, index=adata.obs_names),
    ],
):
    mask = mask.values
    if label != "All cells":
        ax.scatter(
            umap_xy[mask, 0], umap_xy[mask, 1],
            c="#DDDDDD", s=3, linewidths=0, rasterized=True, zorder=1,
        )
    scplot = ax.scatter(
        umap_xy[mask, 0], umap_xy[mask, 1],
        c=dpt[mask],
        cmap="viridis_r",
        vmin=0, vmax=1,
        s=8, linewidths=0, rasterized=True, zorder=2,
    )
    plt.colorbar(scplot, ax=ax, shrink=0.75, label="Pseudotime")
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.set_xlabel("UMAP 1", fontsize=9)
    ax.set_ylabel("UMAP 2", fontsize=9)
    remove_spines(ax)

fig.suptitle(
    "Diffusion pseudotime rooted in MGE progenitors",
    fontsize=12, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig("dpt_umap_split.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: dpt_umap_split.png")

## 10.5 PAGA connectivity and fate groups

We next:

- ensure neighbors and PAGA are computed at the cluster level,
- keep DPT for pseudotime colouring,
- then draw custom PAGA plots where node colors reflect interneuron fate groups.

In [ ]:
# Ensure neighbors + PAGA at cluster level
if "connectivities" not in adata.obsp:
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=30, n_pcs=50)
sc.tl.paga(adata, groups="cluster_name")

# DPT is already computed above, but re-run to be safe if needed
if "dpt_pseudotime" not in adata.obs.columns:
    sc.tl.diffmap(adata, n_comps=15)
    root_mask = adata.obs["cluster_name"].isin(ROOT_CLUSTERS)
    root_idx = np.where(root_mask)[0]
    adata.uns["iroot"] = int(root_idx[np.argmin(adata.obsm["X_diffmap"][root_idx, 1])])
    sc.tl.dpt(adata, n_dcs=10)

## 10.6 Helper: PAGA plotting with fate colours

We define a helper function `draw_paga_fate` that:

- runs PAGA for a given subset,
- draws edges with thickness proportional to connectivity,
- colours nodes by fate group,
- adds cluster labels with a white outline for readability.

In [ ]:
import scipy.sparse as sp
import networkx as nx

def draw_paga_fate(
    adata_sub,
    ax,
    title,
    threshold=0.10,
    edge_width_max=6.0,
    node_size=800,
    font_size=9,
):
    """
    Draw PAGA graph with:
      - Nodes coloured by fate group (PV/SST/Reln/LGE)
      - Edge thickness scaled by connectivity weight
      - Edges hidden below threshold (avoids dark tangle)
      - White outline on labels for readability
    """
    # Run PAGA on this subset
    if "connectivities" not in adata_sub.obsp:
        sc.pp.neighbors(adata_sub, use_rep="X_pca", n_neighbors=20)
    sc.tl.paga(adata_sub, groups="cluster_name")

    conn = adata_sub.uns["paga"]["connectivities"]
    if sp.issparse(conn):
        conn = conn.toarray()
    conn = np.array(conn)

    cats = list(adata_sub.obs["cluster_name"].cat.categories)

    # Build graph
    G = nx.Graph()
    for i, cl in enumerate(cats):
        G.add_node(cl)
    for i in range(len(cats)):
        for j in range(i + 1, len(cats)):
            w = conn[i, j]
            if w > threshold:
                G.add_edge(cats[i], cats[j], weight=float(w))

    # Spring layout weighted by connectivity
    if len(G.edges) > 0:
        pos = nx.spring_layout(G, weight="weight", seed=42, k=2.5)
    else:
        pos = nx.spring_layout(G, seed=42, k=2.5)

    # Draw edges
    for i in range(len(cats)):
        for j in range(i + 1, len(cats)):
            w = conn[i, j]
            if w < threshold:
                continue
            x0, y0 = pos[cats[i]]
            x1, y1 = pos[cats[j]]
            lw = w / 0.5 * edge_width_max
            ax.plot(
                [x0, x1],
                [y0, y1],
                color="#444444",
                linewidth=lw,
                alpha=min(0.4 + w, 0.85),
                solid_capstyle="round",
                zorder=1,
            )

    # Draw nodes
    for cl in cats:
        fate = FATE_MAP.get(cl, "unknown")
        color = FATE_COLORS.get(fate, "#AAAAAA")
        x, y = pos[cl]
        ax.scatter(
            x,
            y,
            s=node_size,
            c=color,
            edgecolors="white",
            linewidths=1.5,
            zorder=3,
        )

    # Labels with white outline
    for cl in cats:
        x, y = pos[cl]
        short = (
            cl.replace("mge_prog_", "prog:")
              .replace("mge_neu_", "neu:")
              .replace("cge_prog_", "cprog:")
              .replace("cge_neu_", "cneu:")
              .replace("lge_prog_", "lprog:")
              .replace("lge_neu_", "lneu:")
        )
        fate = FATE_MAP.get(cl, "unknown")
        color = FATE_COLORS.get(fate, "#AAAAAA")

        txt = ax.text(
            x,
            y - 0.12,
            short,
            ha="center",
            va="top",
            fontsize=font_size,
            fontweight="bold",
            color=color,
            zorder=4,
        )
        txt.set_path_effects(
            [pe.withStroke(linewidth=2.5, foreground="white")]
        )

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.5, 1.4)
    ax.axis("off")

## 10.7 PAGA with fate colours (all cells)

We first draw a PAGA graph using all cells, colouring nodes by fate group. This links the graph abstraction directly to PV/SST/Reln/LGE biology.

In [ ]:
print("\nPlotting PAGA with fate colours (all cells)...")

fig, ax = plt.subplots(figsize=(11, 9), facecolor="white")
draw_paga_fate(
    adata,
    ax,
    title="PAGA cluster connectivity — coloured by interneuron fate",
    threshold=0.10,
    edge_width_max=6.0,
    node_size=900,
    font_size=9,
)

legend_patches = [
    mpatches.Patch(color=col, label=fate)
    for fate, col in FATE_COLORS.items()
]
ax.legend(
    handles=legend_patches,
    frameon=False,
    fontsize=9,
    loc="lower left",
    title="Fate group",
    title_fontsize=9,
    bbox_to_anchor=(0, 0),
)

plt.tight_layout()
plt.savefig("paga_fate_colours.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: paga_fate_colours.png")

## 10.8 PAGA comparison: WT vs KO with fate colours

We now draw PAGA graphs for WT and KO separately, keeping node colours as fate groups. This allows direct visual comparison of connectivity patterns for each fate group between genotypes.

In [ ]:
print("\nPlotting WT vs KO PAGA with fate colours...")

fig, axes = plt.subplots(1, 2, figsize=(20, 9), facecolor="white")

for ax, cond in zip(axes, [ref_group, test_group]):
    sub = adata[adata.obs[condition_key] == cond].copy()
    sub.obs["cluster_name"] = pd.Categorical(
        sub.obs["cluster_name"], categories=cluster_order, ordered=True
    )
    draw_paga_fate(
        sub,
        ax,
        title=f"PAGA — {cond}",
        threshold=0.08,
        edge_width_max=7.0,
        node_size=800,
        font_size=8.5,
    )

legend_patches = [
    mpatches.Patch(color=col, label=fate)
    for fate, col in FATE_COLORS.items()
]
fig.legend(
    handles=legend_patches,
    frameon=False,
    fontsize=9,
    loc="lower center",
    title="Fate group",
    title_fontsize=9,
    ncol=3,
    bbox_to_anchor=(0.5, -0.01),
)

fig.suptitle(
    "PAGA comparison: WT vs Klf13 KO\n"
    "Node colour = interneuron fate group · Edge thickness = transition probability",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig("paga_fate_wt_vs_ko.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: paga_fate_wt_vs_ko.png")

## 10.9 Fate-group level PAGA

Finally, we collapse the 14 clusters into 6 fate groups and rerun PAGA at the fate-group level. This gives a very compact “big picture” of PV/SST/Reln/LGE connectivity and how it shifts in Klf13 KO.

Node size is proportional to cell count per fate group; edge width and labels show PAGA connectivity weights.

In [ ]:
print("\nPlotting fate-group level PAGA...")

# Ensure fate_group categorical order
adata.obs["fate_group"] = adata.obs["cluster_name"].map(FATE_MAP).fillna("unknown")
adata.obs["fate_group"] = pd.Categorical(
    adata.obs["fate_group"],
    categories=list(FATE_COLORS.keys()),
    ordered=False,
)

import scipy.sparse as sps
import networkx as nx

fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor="white")

for ax, cond in zip(axes, [ref_group, test_group]):
    sub = adata[adata.obs[condition_key] == cond].copy()
    sub.obs["fate_group"] = sub.obs["cluster_name"].map(FATE_MAP).fillna("unknown")
    sub.obs["fate_group"] = pd.Categorical(
        sub.obs["fate_group"],
        categories=list(FATE_COLORS.keys()),
        ordered=False,
    )

    sc.tl.paga(sub, groups="fate_group")

    conn_s = sub.uns["paga"]["connectivities"]
    if sps.issparse(conn_s):
        conn_s = conn_s.toarray()
    conn_s = np.array(conn_s)
    cats_s = list(sub.obs["fate_group"].cat.categories)

    G = nx.Graph()
    threshold = 0.05
    for i, f in enumerate(cats_s):
        G.add_node(f)
    for i in range(len(cats_s)):
        for j in range(i + 1, len(cats_s)):
            w = conn_s[i, j]
            if w > threshold:
                G.add_edge(cats_s[i], cats_s[j], weight=float(w))

    pos = nx.spring_layout(G, weight="weight", seed=42, k=3.0)

    # Draw edges
    for i in range(len(cats_s)):
        for j in range(i + 1, len(cats_s)):
            w = conn_s[i, j]
            if w < threshold:
                continue
            x0, y0 = pos[cats_s[i]]
            x1, y1 = pos[cats_s[j]]
            lw = w / 0.3 * 10.0
            ax.plot(
                [x0, x1],
                [y0, y1],
                color="#555555",
                linewidth=lw,
                alpha=min(0.3 + w * 1.5, 0.9),
                solid_capstyle="round",
                zorder=1,
            )
            mx, my = (x0 + x1) / 2, (y0 + y1) / 2
            ax.text(
                mx,
                my,
                f"{w:.2f}",
                ha="center",
                va="center",
                fontsize=7.5,
                color="#333333",
                path_effects=[pe.withStroke(linewidth=2, foreground="white")],
            )

    # Draw nodes
    for fate in cats_s:
        color = FATE_COLORS.get(fate, "#AAAAAA")
        x, y = pos[fate]
        n_cells = (sub.obs["fate_group"] == fate).sum()
        sz = max(300, min(3000, n_cells * 3))
        ax.scatter(
            x,
            y,
            s=sz,
            c=color,
            edgecolors="white",
            linewidths=2.5,
            zorder=3,
        )
        txt = ax.text(
            x,
            y,
            fate,
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
            color="white",
            zorder=4,
        )
        txt.set_path_effects(
            [pe.withStroke(linewidth=2.5, foreground=color)]
        )

    ax.set_title(
        f"Fate-group PAGA — {cond}\n(node size = cell count · edge weight shown)",
        fontsize=11,
        fontweight="bold",
    )
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.axis("off")

fig.suptitle(
    "Fate-group level PAGA: PV / SST / Reln / LGE connectivity\nWT vs Klf13 KO comparison",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("paga_fate_group_level.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: paga_fate_group_level.png")

print("\nAll PAGA fate plots saved:")
for f in [
    "paga_fate_colours.png",
    "paga_fate_wt_vs_ko.png",
    "paga_fate_group_level.png",
]:
    print(" ", f)

## 10.10 Save AnnData with DPT and fate PAGA annotations

We finally save the GABA AnnData object with DPT pseudotime, cluster-level PAGA, and fate-group annotations for reproducibility and downstream figure generation.

In [ ]:
output_file = "gaba_trajectory_dpt_paga_fate.h5ad"
adata.write(output_file)
print(f"Saved trajectory + fate-PAGA AnnData to: {output_file}")